In [3]:
%pip install highlight_text

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# pandas: DataFrame loading, filtering, and per-90 computation
# matplotlib: figure rendering and global tick/label color configuration
# seaborn: sns.swarmplot() — the core function for beeswarm distribution plots
# highlight_text: extends matplotlib with inline multi-color text annotations
#   (not used in the final plot here but available for styled title/label text)
# matplotlib.font_manager: used below to enumerate all registered system fonts

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import highlight_text
import matplotlib.font_manager
from IPython.core.display import HTML

# Utility to render the full list of available matplotlib fonts in the notebook output.
# Useful for selecting typography for chart annotations. Not part of the analysis pipeline.
def make_html(fontname):
    return "<p>{font}: <span style='font-family: {font}'; font-size: 24px;'>{font}</p>".format(font=fontname)

code = "\n".join([make_html(font) for font in sorted(set([f.name for f in matplotlib.font_manager.fontManager.ttflist]))])

# Load the FBref progressive pass dataset for the 2020-21 Premier League season.
# Columns:
#   Player — name in "Full Name\url-slug" format (cleaned in the next cell)
#   Pos    — player position (DF, MF, FW, GK)
#   Squad  — club name
#   90s    — 90-minute equivalents played (float)
#   Prog   — total progressive passes completed (integer count)
df = pd.read_csv('data/beeswarmTutorial11.csv')

# Chart theme constants — defined once and reused across all plot elements.
text_color = 'white'
background = '#313332'

df.head()

In [ ]:
# Normalize total progressive passes by minutes played to obtain a per-90 rate.
# Dividing raw Prog count by 90s removes the playing-time bias: a player with
# 50 progressive passes in 10 appearances (900 mins) and a player with 100 in 20
# appearances (1800 mins) have the same per90 rate (5.0), so their positions
# on the beeswarm x-axis will be identical — reflecting equal efficiency.
df['per90'] = df['Prog'] / df['90s']
df

In [ ]:
# Apply a minimum playing time threshold of 6.5 90-minute equivalents (~585 minutes).
# Without this filter, players with very few appearances produce extreme per90 rates
# that are statistical outliers, not genuine performance signals. For example, a
# substitute who completes 5 progressive passes in 10 minutes would sit at ~45 per90,
# dwarfing any legitimate player. The 6.5 threshold corresponds to roughly 7 full games.
# reset_index() resets row indices after the filter for clean downstream access.
df = df[df['90s'] >= 6.5].reset_index()
df.describe()

In [ ]:
# Sort by per90 descending to inspect the top progressive passers in the league.
# Remove goalkeepers (Pos == 'GK') from the analysis — goalkeepers rarely attempt
# progressive passes in the FBref definition (advancing the ball 10+ yards toward goal
# from outside the final third) and their presence at the low end of the distribution
# would compress the scale for outfield players.
df = df.sort_values(by='per90', ascending=False)
df = df[df['Pos'] != 'GK']
df.head()

In [ ]:
# Set up the dark-themed figure with consistent styling.
fig, ax = plt.subplots(figsize=(10, 5))
fig.set_facecolor(background)
ax.patch.set_facecolor(background)

mpl.rcParams['xtick.color'] = text_color
mpl.rcParams['ytick.color'] = text_color

ax.grid(ls='dotted', lw=.5, color='lightgrey', axis='y', zorder=1)
for x in ['top', 'bottom', 'left', 'right']:
    ax.spines[x].set_visible(False)

# sns.swarmplot() renders a 1D beeswarm (strip) chart along the x-axis.
# Each dot represents one player positioned at their per90 value.
# The "swarm" algorithm displaces overlapping points vertically to prevent occlusion,
# revealing the full density and shape of the distribution — unlike a histogram,
# every individual data point remains visible and identifiable.
# x='per90': the continuous metric plotted on the horizontal axis
# color='white': all dots rendered in the same color; highlighted players are overlaid
# zorder=1: swarm layer sits beneath the highlighted player markers
sns.swarmplot(x='per90', data=df, color='white', zorder=1)

# Highlight specific players of interest by overlaying a larger, colored scatter marker
# at their exact per90 coordinate. y=0 is correct here because swarmplot uses a single
# categorical axis (all points are on the same horizontal row). zorder=2 ensures
# highlighted markers render above the white swarm cloud.
# plt.text() labels each highlighted point just below (y=-0.002 offset) for readability.

# Thiago Alcántara: highest progressive pass rate in the league at ~9.88 per90
plt.scatter(x=9.87, y=0, c='red', edgecolor='white', s=200, zorder=2)
plt.text(s='Thiago', x=9.87, y=-0.002, c=text_color)

# De Bruyne: top-tier rate at ~7.65 per90, used as a second reference point
plt.scatter(x=7.654, y=0, c='blue', edgecolor='white', s=200, zorder=2)
plt.text(s='De Bruyne', x=7.564, y=-0.002, c=text_color)

plt.title('Progressive passes per 90 — Premier League 2020-21', c=text_color, fontsize=14)
plt.xlabel('Progressive passes per 90 minutes', c=text_color)

## Summary: Beeswarm Plot — Progressive Pass Distribution

### What This Notebook Does

This notebook produces a beeswarm plot showing the distribution of progressive passes per 90 minutes across all outfield Premier League players in the 2020-21 season with at least 6.5 90-minute equivalents played. Two specific players (Thiago Alcântara and Kevin De Bruyne) are highlighted against the full population to contextualize their statistical outlier status.

### Key Concepts

- **Progressive pass (FBref definition)**: A completed pass that moves the ball at least 10 yards toward the opponent's goal from outside the final third, or any pass into the final third or penalty area. This metric captures ball-progression intent and execution — it differentiates players who advance the team up the pitch from those who maintain possession laterally or backward.
- **Per-90 normalization** (`Prog / 90s`): Dividing the raw progressive pass count by the player's 90-minute equivalent appearances equalizes for playing time. Without this, players who simply played more minutes accumulate more progressive passes regardless of efficiency.
- **`sns.swarmplot()`**: Seaborn's beeswarm implementation distributes points along a 1D axis (x='per90') using an algorithm that minimizes vertical displacement while guaranteeing no two points overlap. The result is a non-parametric density visualization where every data point is visible individually — unlike a histogram (which aggregates into bins and loses individual identity) or a KDE (which smooths and may obscure outliers).
- **Minimum threshold (`90s >= 6.5`)**: ~585 minutes is the threshold below which per-90 rates become statistically unreliable. A player with one dominant game in limited minutes would rank artificially at the top of the chart. The threshold is intentionally conservative — raising it to 10+ 90s would further reduce noise at the cost of excluding rotation players.
- **Overlay technique**: Highlighted players are added using `plt.scatter()` at `y=0` (the center of the swarm's y-axis) with `zorder=2`. Because `swarmplot` uses a single categorical y-axis, all players technically sit at `y=0` in data coordinates — the vertical spread is purely for visual non-overlap and has no data meaning.

### Data Available

| Object | Description |
|---|---|
| `df` | Filtered DataFrame of outfield PL players with 6.5+ 90s, including `per90` column |
| `text_color`, `background` | Chart theme constants for consistent styling |

### Ideas to Extract More Value

- **Position-stratified beeswarm**: Add `hue='Pos'` to `sns.swarmplot()` to color-code dots by position. This immediately reveals that midfielders cluster at higher per90 rates than defenders or forwards, reflecting their role in ball progression.
- **Median and percentile annotations**: Use `ax.axvline()` to draw vertical reference lines at the 25th, 50th, and 75th percentiles of the distribution. This gives every dot an immediate positional context relative to the full league.
- **Club-level aggregation**: Group `df` by `Squad`, compute the team mean `per90`, and overlay club average markers on the beeswarm. This adds a team layer to the player-level view.
- **Multi-metric beeswarm panel**: Create a `plt.subplots(1, N)` grid with one beeswarm per metric (progressive passes, key passes, shot-creating actions, etc.) and highlight the same player across all panels. This builds a multi-dimensional profile from a series of 1D distributions.
- **Interactive version**: Replace `matplotlib` with `plotly.express.strip()` (Plotly's beeswarm equivalent) to produce an interactive chart where hovering a dot reveals the player name, club, and exact per90 value — useful for exploratory analysis.